# Stage 4 — baseline training and monitoring

Run this notebook with your local VS Code Python kernel and CUDA-enabled PyTorch. It finds CIFAR-100 under the project `data/` folder or `/data`, and keeps checkpoints, splits, and results in the project folder.

In [ ]:
from pathlib import Path
import csv
import os
import subprocess
import sys
import tarfile

import matplotlib.pyplot as plt
import torch

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'src' / 'train.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the project folder in VS Code so this notebook can find src/train.py.')
sys.path.insert(0, str(ROOT))
print('Project files:', ROOT)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Install CUDA-enabled PyTorch and select your local GPU interpreter.'

In [ ]:
OUTPUT_ROOT = ROOT
SPLIT_DIR = OUTPUT_ROOT / 'splits'
PROJECT_DATA = ROOT / 'data'
archive_name = 'cifar-100-python.tar.gz'
search_roots = [PROJECT_DATA, Path('/data'), Path('C:/data')]

def find_cifar_folder(roots):
    for base in roots:
        if base.exists():
            for train_file in base.rglob('train'):
                if train_file.is_file() and (train_file.parent / 'test').is_file():
                    return train_file.parent
    return None

dataset_dir = find_cifar_folder(search_roots)
archives = [base / archive_name for base in search_roots]
archive = next((path for path in archives if path.is_file()), None)
if dataset_dir is None and archive is not None:
    with tarfile.open(archive, 'r:gz') as tar:
        tar.extractall(archive.parent)
    dataset_dir = find_cifar_folder([archive.parent])
if dataset_dir is None:
    raise FileNotFoundError(
        f'Could not find CIFAR train/test files. Upload {archive_name} or the extracted '
        f'cifar-100-python folder under /data or the project data/ folder. '
        f'Checked folders: {search_roots}; archives: {archives}'
    )
DATA_ROOT = dataset_dir.parent  # torchvision expects DATA_ROOT/cifar-100-python
print('CIFAR-100 files found at:', dataset_dir)
print('Data root:', DATA_ROOT)
print('Artifacts will be saved under:', OUTPUT_ROOT)

## W&B monitoring

The next cell asks for your W&B API key using hidden input. The key is kept in the runtime environment and is not written into this notebook or passed as a command-line argument. Set `USE_WANDB = False` if you want local logs only.

In [ ]:
USE_WANDB = True
WANDB_PROJECT = 'visual-similarity-search'
if USE_WANDB:
    try:
        import wandb
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'], check=True)
    from getpass import getpass
    api_key = getpass('W&B API key (hidden): ')
    if not api_key:
        raise ValueError('Enter a W&B API key or set USE_WANDB = False.')
    os.environ['WANDB_API_KEY'] = api_key
    del api_key
    print('W&B key is set for this runtime.')

## Run configuration

The recovery run uses Adam at lr 0.0003, zero weight decay, and batch-all mining for 3 epochs before batch-hard. Test evaluation is skipped while tuning. The run stops at epoch 3 unless the stabilization checks pass.

In [ ]:
EPOCHS = 40
RUN_COMPARISON = False
SKIP_TEST = True
BASELINE_LR = 3e-4
WEIGHT_DECAY = 0.0
MARGIN = 0.3
EMBEDDING_DIM = 128
P, K = 32, 8
BATCH_ALL_EPOCHS = 3
SEED = 42
NUM_WORKERS = 2

command = [
    sys.executable, '-m', 'src.train',
    '--epochs', str(EPOCHS), '--seed', str(SEED),
    '--weight-decay', str(WEIGHT_DECAY), '--margin', str(MARGIN),
    '--embedding-dim', str(EMBEDDING_DIM), '--P', str(P), '--K', str(K),
    '--batch-all-epochs', str(BATCH_ALL_EPOCHS),
    '--num-workers', str(NUM_WORKERS), '--data-root', str(DATA_ROOT),
    '--split-dir', str(SPLIT_DIR), '--output-root', str(OUTPUT_ROOT),
    '--no-download',
]
if RUN_COMPARISON:
    command.extend(['--compare', '--lr', str(BASELINE_LR)])
else:
    command.extend(['--optimizer', 'adam', '--lr', str(BASELINE_LR)])
if SKIP_TEST:
    command.append('--skip-test')
if USE_WANDB:
    command.extend(['--wandb', '--wandb-project', WANDB_PROJECT])
    help_result = subprocess.run(
        [sys.executable, '-m', 'src.train', '--help'], cwd=ROOT,
        capture_output=True, text=True
    )
    required_flags = ('--wandb', '--wandb-project', '--batch-all-epochs')
    if help_result.returncode or any(flag not in help_result.stdout for flag in required_flags):
        raise RuntimeError(
            f'{ROOT / "src"} is an older copy. Upload the current src/ folder, '
            'then rerun this cell.'
        )
print(' '.join(command))

In [ ]:
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(
    command, cwd=ROOT, env=env, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='', flush=True)
status = process.wait()
if status:
    raise RuntimeError(f'Training exited with code {status}. Check the last epoch output above.')
print('Training complete.')

In [ ]:
def plot_run(run_name):
    path = OUTPUT_ROOT / 'runs' / run_name / 'metrics.csv'
    with path.open(newline='') as f:
        rows = list(csv.DictReader(f))
    epochs = [int(row['epoch']) for row in rows]
    def series(key):
        return [float(row[key]) for row in rows]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes[0, 0].plot(epochs, series('train_loss')); axes[0, 0].set_title('Train loss')
    axes[0, 1].plot(epochs, series('train_d_ap'), label='d_ap')
    axes[0, 1].plot(epochs, series('train_d_an'), label='d_an')
    axes[0, 1].plot(epochs, series('mean_pairwise_distance'), label='all pairs')
    axes[0, 1].legend(); axes[0, 1].set_title('Train distances')
    axes[1, 0].plot(epochs, series('active_fraction')); axes[1, 0].set_title('Active triplets')
    axes[1, 1].plot(epochs, series('val_r1'), label='R@1')
    axes[1, 1].plot(epochs, series('val_triplet_acc'), label='Random triplet')
    axes[1, 1].plot(epochs, series('val_batch_hard_acc'), label='Batch-hard'); axes[1, 1].legend()
    axes[1, 1].set_title('Validation')
    for ax in axes.flat: ax.set_xlabel('Epoch'); ax.grid(alpha=0.25)
    fig.tight_layout()
    out = OUTPUT_ROOT / 'runs' / run_name / 'curves.png'
    fig.savefig(out, dpi=160)
    plt.show()
    print('Saved', out)

for name in (['baseline', 'nadam', 'sgd'] if RUN_COMPARISON else ['baseline']):
    if (OUTPUT_ROOT / 'runs' / name / 'metrics.csv').exists():
        plot_run(name)

In [ ]:
table = OUTPUT_ROOT / 'results' / 'optimizer_comparison.csv'
if table.exists():
    with table.open(newline='') as f:
        for row in csv.DictReader(f):
            print(row)
else:
    print('Run with RUN_COMPARISON = True to generate the comparison table.')